# CNIBP Result Analysis

独立结果分析笔记本。

- 不修改训练笔记本
- 不重新训练
- 直接对已有 `run_dir` 做误差分层分析


In [ ]:
# 只需要改这4项
GIT_REPO = 'https://github.com/67vmg9wrfn-beep/Lab.git'
GIT_BRANCH = 'main'
PROJECT_SUBDIR = '06_experiments/cnibp/repro_ppg_bp'
RUN_DIR = '/content/drive/MyDrive/cnibp_repro_outputs/run_20260305_144740'


In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
print('[OK] drive mounted')


In [ ]:
import os, shutil, subprocess

if os.path.exists('/content/repo_src'):
    shutil.rmtree('/content/repo_src')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_BRANCH, GIT_REPO, '/content/repo_src'], check=True)
subprocess.run(['git', '-C', '/content/repo_src', 'rev-parse', '--short', 'HEAD'], check=True)
PROJECT_ROOT = f'/content/repo_src/{PROJECT_SUBDIR}'
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
import os, subprocess

env = os.environ.copy()
env['PYTHONPATH'] = f"{PROJECT_ROOT}/src:" + env.get('PYTHONPATH', '')
cmd = [
    'python', '-m', 'cnibp_repro.analyze_run',
    '--run_dir', RUN_DIR,
]
print('RUN CMD:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)


In [ ]:
from pathlib import Path
import pandas as pd
import json

analysis_dir = Path(RUN_DIR) / 'analysis'
print('analysis_dir =', analysis_dir)

overall_path = analysis_dir / 'overall_error_summary.json'
range_path = analysis_dir / 'global_range_error_summary.csv'
fold_path = analysis_dir / 'fold_level_error_summary.csv'

print('\n===== overall_error_summary.json =====')
print(json.dumps(json.loads(overall_path.read_text()), indent=2, ensure_ascii=False))

print('\n===== global_range_error_summary.csv =====')
display(pd.read_csv(range_path))

print('\n===== fold_level_error_summary.csv =====')
display(pd.read_csv(fold_path))
